# Download Evaluation Datasets to Google Drive

Downloads directly from Colab (datacenter internet) to Drive. No local wifi needed.

**Datasets (eval-only where applicable):**

| # | Dataset | Size | Source |
|---|---|---|---|
| 1 | ASVspoof 2019 LA eval | Already on Drive | - |
| 2 | ASVspoof 2021 LA eval | ~7.8 GB | Zenodo |
| 3 | ASVspoof 2021 DF eval | ~34.5 GB (4 parts) | Zenodo |
| 4 | In-The-Wild | Already on Drive | - |
| 5 | ASVspoof 5 eval | ~85 GB (11 parts) + protocols | Zenodo |
| 6 | MLAAD (subset) | TBD (gated) | HuggingFace |
| 7 | CodecFake+ | ~101 GB full; CoSG eval = 244 MB | HuggingFace |
| 8 | Deepfake-Eval-2024 | TBD (gated) | HuggingFace |

**Run cells selectively** -- no need to download everything at once.

## 0. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Base directory for all eval datasets
BASE = '/content/drive/MyDrive/Jakob/Datasets'
os.makedirs(BASE, exist_ok=True)

# Helper: download with resume support
def dl(url, dest_dir):
    """Download file to dest_dir with wget, resuming partial downloads."""
    os.makedirs(dest_dir, exist_ok=True)
    !wget -c -P {dest_dir} "{url}"

print(f'Base dir: {BASE}')
!df -h /content/drive | tail -1

Mounted at /content/drive
Base dir: /content/drive/MyDrive/Jakob/Datasets
drive           226G   32G  195G  14% /content/drive


## 1. ASVspoof 2021 LA eval (~7.8 GB)

Source: https://zenodo.org/records/4837263

In [ ]:
ASV21_LA = f'{BASE}/ASVspoof2021_LA'
!ls {ASV21_LA}

ASVspoof2021_LA_eval  ASVspoof2021_LA_eval.tar.gz  LA-keys-full.tar.gz


### Download ASVspoof 2021 LA eval keys/protocols

Keys are separate from audio. Get them from the ASVspoof website.

In [ ]:
dl('https://www.asvspoof.org/asvspoof2021/LA-keys-full.tar.gz', ASV21_LA)

--2026-02-24 04:00:12--  https://www.asvspoof.org/asvspoof2021/LA-keys-full.tar.gz
Resolving www.asvspoof.org (www.asvspoof.org)... 193.55.113.222
Connecting to www.asvspoof.org (www.asvspoof.org)|193.55.113.222|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21235082 (20M) [application/x-gzip]
Saving to: ‘/content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_LA/LA-keys-full.tar.gz’

LA-keys-full.tar.gz 100%[===================>]  20.25M  5.18MB/s    in 5.3s    

2026-02-24 04:00:18 (3.84 MB/s) - ‘/content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_LA/LA-keys-full.tar.gz’ saved [21235082/21235082]



## 2. ASVspoof 2021 DF eval (~34.5 GB)

Source: https://zenodo.org/records/4835108

4 parts, each ~8.6 GB. Download then concatenate and extract.

In [ ]:
# Concatenate and extract on local SSD (10-50x faster than Drive)
print('Concatenating parts on local SSD...')
TMP = '/content/tmp_asv21df'
os.makedirs(TMP, exist_ok=True)
!cp /content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_DF/ASVspoof2021_DF_eval_part00.tar.gz {TMP}
!cp /content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_DF/ASVspoof2021_DF_eval_part01.tar.gz {TMP}
!cp /content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_DF/ASVspoof2021_DF_eval_part02.tar.gz {TMP}
!cp /content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_DF/ASVspoof2021_DF_eval_part03.tar.gz {TMP}
!cat {TMP}/ASVspoof2021_DF_eval_part0*.tar.gz > {TMP}/full.tar.gz

print('Extracting on local SSD...')
!tar -xzf {TMP}/full.tar.gz -C {TMP}

print('Moving extracted audio to Drive...')
!mv {TMP}/ASVspoof2021_DF_eval {ASV21_DF}/
# Or if the tar extracts to a different folder name, adjust above

print('Cleaning up local SSD...')
!rm -rf {TMP}

print('Done.')
!ls {ASV21_DF}

Concatenating parts on local SSD...
Extracting on local SSD...
Moving extracted audio to Drive...
mv: cannot stat '{TMP}/ASVspoof2021_DF_eval': No such file or directory
Cleaning up local SSD...
Done.
ls: cannot access '{ASV21_DF}': No such file or directory


## 3. ASVspoof 5 eval (~85 GB)

Source: https://zenodo.org/records/14498691

11 tar files (`flac_E_aa` through `flac_E_aj` + `flac_E_ak` if exists) + protocols.

**This is the biggest download. Run overnight if needed.**

In [ ]:
ASV5 = f'{BASE}/ASVspoof5'
os.makedirs(ASV5, exist_ok=True)

# Protocols first (21 MB) -- always download this
dl('https://zenodo.org/records/14498691/files/ASVspoof5_protocols.tar.gz', ASV5)
!tar -xzf {ASV5}/ASVspoof5_protocols.tar.gz -C {ASV5}
print('Protocols extracted.')
!ls {ASV5}

--2026-02-24 05:03:34--  https://zenodo.org/records/14498691/files/ASVspoof5_protocols.tar.gz
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 188.184.98.114, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 416 REQUESTED_RANGE_NOT_SATISFIABLE

    The file is already fully retrieved; nothing to do.

Protocols extracted.
ASVspoof5.codec.config.csv	   ASVspoof5.train.tsv
ASVspoof5.dev.track_1.tsv	   flac_E_aa.tar
ASVspoof5.dev.track_2.enroll.tsv   flac_E_ab.tar
ASVspoof5.dev.track_2.trial.tsv    flac_E_ac.tar
ASVspoof5.eval.track_1.tsv	   flac_E_ae.tar
ASVspoof5.eval.track_2.enroll.tsv  flac_E_ah.tar
ASVspoof5.eval.track_2.trial.tsv   flac_E_ai.tar
ASVspoof5_protocols.tar.gz


In [ ]:
# Download eval tars to local SSD first, then extract to Drive
TMP5 = '/content/tmp_asv5'
os.makedirs(TMP5, exist_ok=True)

eval_parts = [f'flac_E_a{chr(ord("a")+i)}' for i in range(10)]  # aa through aj
print(f'Will download {len(eval_parts)} files: {eval_parts}')

for part in eval_parts:
    url = f'https://zenodo.org/records/14498691/files/{part}.tar'
    local_dest = f'{TMP5}/{part}.tar'
    drive_dest = f'{ASV5}/{part}.tar'
    # Skip if already on Drive
    if os.path.exists(drive_dest):
        size_gb = os.path.getsize(drive_dest) / 1e9
        print(f'  {part}.tar already on Drive ({size_gb:.1f} GB), skipping')
        continue
    print(f'\n--- Downloading {part}.tar ---')
    dl(url, TMP5)
    # Extract on local SSD, move to Drive, delete tar
    print(f'Extracting {part}.tar on local SSD...')
    !tar -xf {local_dest} -C {TMP5}
    print(f'Moving to Drive...')
    !rsync -a {TMP5}/flac/ {ASV5}/flac/ 2>/dev/null; mv {TMP5}/flac_E* {ASV5}/ 2>/dev/null || true
    !rm -f {local_dest}
    # Clean extracted files from local to free space for next part
    !find {TMP5} -name "*.flac" -delete 2>/dev/null
    print(f'  {part} done.')

Will download 10 files: ['flac_E_aa', 'flac_E_ab', 'flac_E_ac', 'flac_E_ad', 'flac_E_ae', 'flac_E_af', 'flac_E_ag', 'flac_E_ah', 'flac_E_ai', 'flac_E_aj']
  flac_E_aa.tar already on Drive (1.3 GB), skipping
  flac_E_ab.tar already on Drive (0.0 GB), skipping
  flac_E_ac.tar already on Drive (0.0 GB), skipping

--- Downloading flac_E_ad.tar ---
--2026-02-24 05:03:48--  https://zenodo.org/records/14498691/files/flac_E_ad.tar
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 188.184.98.114, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8467200000 (7.9G) [application/octet-stream]
Saving to: ‘/content/tmp_asv5/flac_E_ad.tar’

flac_E_ad.tar       100%[===================>]   7.88G  21.7MB/s    in 5m 23s  

2026-02-24 05:09:11 (25.0 MB/s) - ‘/content/tmp_asv5/flac_E_ad.tar’ saved [8467200000/8467200000]

Extracting flac_E_ad.tar on local SSD...
Moving to Drive...
  flac_E_ad done.
  flac_E_

In [ ]:
# Cleanup local temp (extraction was done per-part above)
!rm -rf {TMP5} 2>/dev/null
print('ASVspoof 5 eval audio on Drive:')
!ls {ASV5}
!du -sh {ASV5}

Will download 10 files: ['flac_E_aa', 'flac_E_ab', 'flac_E_ac', 'flac_E_ad', 'flac_E_ae', 'flac_E_af', 'flac_E_ag', 'flac_E_ah', 'flac_E_ai', 'flac_E_aj']

--- Downloading flac_E_aa.tar ---
--2026-02-24 04:03:15--  https://zenodo.org/records/14498691/files/flac_E_aa.tar
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 137.138.52.235, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8449781760 (7.9G) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/Jakob/Datasets/ASVspoof5/flac_E_aa.tar’

flac_E_aa.tar        15%[==>                 ]   1.19G  25.4MB/s    eta 5m 40s ^C

--- Downloading flac_E_ab.tar ---
--2026-02-24 04:04:16--  https://zenodo.org/records/14498691/files/flac_E_ab.tar
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.153.219, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting res

In [ ]:
# # Extract eval audio (run after all parts downloaded)
# # Extract to ephemeral storage if running eval immediately, or to Drive for persistence

# EXTRACT_TO = ASV5  # Change to '/content/data/asvspoof5' for ephemeral

# for part in eval_parts:
#     tar_path = f'{ASV5}/{part}.tar'
#     if os.path.exists(tar_path):
#         print(f'Extracting {part}.tar...')
#         !tar -xf {tar_path} -C {EXTRACT_TO}

# print('Done.')
# !ls {EXTRACT_TO}

## 4. MLAAD (gated -- needs HF login)

Source: https://huggingface.co/datasets/mueller91/MLAAD

This is a **gated dataset** -- you must:
1. Create a HuggingFace account
2. Go to the dataset page and accept the license
3. Generate an access token at https://huggingface.co/settings/tokens

We download a **subset** of languages for eval (not the full 678h).

In [ ]:
!pip install -q huggingface_hub

# Login -- paste your HF token when prompted (hf_...)
!huggingface-cli login


In [ ]:
MLAAD = f'{BASE}/MLAAD'
os.makedirs(MLAAD, exist_ok=True)

# Clone the repo (git-lfs) -- this gets EVERYTHING
# For a subset, we can use HF's API to download specific folders
from huggingface_hub import snapshot_download

# Download just a few languages for eval
# Adjust allow_patterns to pick languages you want
LANGUAGES = ['en_US', 'en_UK', 'de_DE', 'fr_FR', 'es_ES', 'zh_CN', 'ru_RU', 'it_IT', 'uk_UA', 'pl_PL']
patterns = []
for lang in LANGUAGES:
    patterns.append(f'fake/{lang}/**')

print(f'Downloading MLAAD subset: {LANGUAGES}')
print(f'Patterns: {patterns}')

snapshot_download(
    repo_id='mueller91/MLAAD',
    repo_type='dataset',
    local_dir=MLAAD,
    allow_patterns=patterns,
)

print('\nDone. Contents:')
!ls {MLAAD}/fake/ 2>/dev/null || echo 'Check folder structure'

## 5. CodecFake+

Source: https://huggingface.co/datasets/CodecFake/CodecFake_Plus_Dataset

Two subsets:
- **CoSG** (Codec Speech Generation): 244 MB -- eval-focused, modern TTS systems. **Download this first.**
- **CoRS** (Codec Re-Synthesis): ~100 GB in 4 parts -- codec encode-decode loops. Optional.

CoSG contains samples from 17 advanced codec-based speech generation models (VALL-E, MaskGCT, CosyVoice, etc.).

In [ ]:
# CodecFake+ goes to shared drive (separate storage quota)
CF = '/content/drive/Shareddrives/Jakob-298-datasets/CodecFakePlus'
os.makedirs(CF, exist_ok=True)

# CoSG eval subset (244 MB) -- the important one
print('Downloading CoSG (244 MB)...')
dl('https://huggingface.co/datasets/CodecFake/CodecFake_Plus_Dataset/resolve/main/CodecFake_plus_CoSG.tar.xz', CF)
dl('https://huggingface.co/datasets/CodecFake/CodecFake_Plus_Dataset/resolve/main/CoSG_labels.txt', CF)

print('\nExtracting CoSG...')
!tar -xJf {CF}/CodecFake_plus_CoSG.tar.xz -C {CF}

print('Done.')
!ls {CF}

In [ ]:
# OPTIONAL: CoRS (codec re-synthesis) -- ~100 GB total
# Downloads to shared drive (same as CoSG above)

DOWNLOAD_CORS = False  # Set to True to download

if DOWNLOAD_CORS:
    # Download parts to local SSD, concat there, then move to Drive
    TMP = '/content/tmp_cors'
    os.makedirs(TMP, exist_ok=True)

    dl('https://huggingface.co/datasets/CodecFake/CodecFake_Plus_Dataset/resolve/main/CoRS_labels.txt', CF)
    for i in range(4):
        print(f'\n--- CoRS part {i}/3 (~25 GB) ---')
        dl(f'https://huggingface.co/datasets/CodecFake/CodecFake_Plus_Dataset/resolve/main/Codecfake_plus_CoRS.part{i}', TMP)

    print('\nConcatenating on local SSD...')
    !cat {TMP}/Codecfake_plus_CoRS.part* > {TMP}/Codecfake_plus_CoRS.tar.xz

    print('Extracting on local SSD...')
    !tar -xJf {TMP}/Codecfake_plus_CoRS.tar.xz -C {TMP}

    print('Moving to shared drive...')
    !rsync -ah --progress {TMP}/CoRS/ {CF}/CoRS/

    print('Cleaning up local SSD...')
    !rm -rf {TMP}
    print('Done.')
else:
    print('Skipping CoRS (100 GB). Set DOWNLOAD_CORS = True to download.')

## 6. Deepfake-Eval-2024 (gated -- needs HF login)

Source: https://huggingface.co/datasets/nuriachandra/Deepfake-Eval-2024

**Gated dataset** -- you must:
1. Go to the dataset page and accept the conditions
2. Be logged in (cell 4.0 above)

Contains 56.5h audio from 88 websites in 52 languages.

In [ ]:
print("just do it manually babe")
# DFE = f'{BASE}/DeepfakeEval2024'
# os.makedirs(DFE, exist_ok=True)

# # First, check what's available after accepting the gate
# from huggingface_hub import list_repo_files

# try:
#     files = list_repo_files('nuriachandra/Deepfake-Eval-2024', repo_type='dataset')
#     print(f'Found {len(files)} files:')
#     for f in files[:30]:  # Show first 30
#         print(f'  {f}')
#     if len(files) > 30:
#         print(f'  ... and {len(files)-30} more')
# except Exception as e:
#     print(f'Error: {e}')
#     print('\nMake sure you:')
#     print('  1. Accepted the license at https://huggingface.co/datasets/nuriachandra/Deepfake-Eval-2024')
#     print('  2. Ran huggingface-cli login above')

In [ ]:
# Download audio subset only
# Adjust allow_patterns based on the file listing above

# from huggingface_hub import snapshot_download

# try:
#     snapshot_download(
#         repo_id='nuriachandra/Deepfake-Eval-2024',
#         repo_type='dataset',
#         local_dir=DFE,
#         allow_patterns=['*audio*', '*.csv', '*.json', '*.txt', 'README*'],  # Adjust after seeing file list
#     )
#     print('Done.')
#     !ls {DFE}
# except Exception as e:
#     print(f'Error: {e}')
#     print('Accept the gate first, then retry.')

## 7. Summary: Check What's on Drive

In [ ]:
print('=== Evaluation Datasets on Drive ===')
print()

datasets = {
    'ASVspoof 2019 LA': '/content/drive/MyDrive/Jakob/Datasets',  # adjust path
    'In-The-Wild': '/content/data/itw',  # adjust path
    'ASVspoof 2021 LA': ASV21_LA if 'ASV21_LA' in dir() else 'not set',
    'ASVspoof 2021 DF': ASV21_DF if 'ASV21_DF' in dir() else 'not set',
    'ASVspoof 5': ASV5 if 'ASV5' in dir() else 'not set',
    'MLAAD': MLAAD if 'MLAAD' in dir() else 'not set',
    'CodecFake+': CF if 'CF' in dir() else 'not set',
    'Deepfake-Eval-2024': DFE if 'DFE' in dir() else 'not set',
}

for name, path in datasets.items():
    if os.path.isdir(path):
        size = !du -sh {path} 2>/dev/null | cut -f1
        n_files = !find {path} -type f 2>/dev/null | wc -l
        print(f'  [x] {name:<25} {size[0]:>8}   ({n_files[0].strip()} files)  {path}')
    else:
        print(f'  [ ] {name:<25} NOT FOUND  {path}')

print()
!df -h /content/drive | tail -1

=== Evaluation Datasets on Drive ===

^C
